<a href="https://colab.research.google.com/github/Peeyusj/bpe-tokenizer/blob/main/bpe_tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
text = "hello😂"
bytes_list = text.encode("utf-8")
print(bytes_list)
print(list(bytes_list))

b'hello\xf0\x9f\x98\x82'
[104, 101, 108, 108, 111, 240, 159, 152, 130]


In [12]:
def get_stats(tokens):
    counts = {}
    for i in range(len(tokens)-1):
        pair = (tokens[i], tokens[i+1])
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# test it
text = "aaabdaaabac"
tokens = list(text.encode("utf-8"))
print(tokens)

stats = get_stats(tokens)
print(stats)

[97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
{(97, 97): 4, (97, 98): 2, (98, 100): 1, (100, 97): 1, (98, 97): 1, (97, 99): 1}


In [13]:
def merge(tokens, pair, new_token):
    new_tokens = []
    i = 0
    while i < len(tokens) - 1:
        if tokens[i] == pair[0] and tokens[i+1] == pair[1]:  # blank 1
            new_tokens.append(new_token)  # blank 2
            i += 2  # blank 3
        else:
            new_tokens.append(tokens[i])
            i += 1
    new_tokens.append(tokens[-1])
    return new_tokens

In [14]:
tokens = list("aaabdaaabac".encode("utf-8"))
print("before:", tokens)

stats = get_stats(tokens)
best_pair = max(stats, key=stats.get)
print("merging:", best_pair)

tokens = merge(tokens, best_pair, 256)
print("after:", tokens)

before: [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
merging: (97, 97)
after: [256, 97, 98, 100, 256, 97, 98, 97, 99]


In [15]:
tokens = list("aaabdaaabac".encode("utf-8"))
merges = {}
num_merges = 20

for i in range(num_merges ):  # blank 1 - how many times?
    stats = get_stats(tokens)
    best_pair = max(stats, key=stats.get)
    new_token = 256 + i  # blank 2 - starts at 256, increases by 1
    tokens = merge(tokens, best_pair, new_token)
    merges[best_pair] = new_token  # saving the rule
    print(f"merge {i+1}: {best_pair} -> {new_token}")

merge 1: (97, 97) -> 256
merge 2: (256, 97) -> 257
merge 3: (257, 98) -> 258
merge 4: (258, 100) -> 259
merge 5: (259, 258) -> 260
merge 6: (260, 97) -> 261
merge 7: (261, 99) -> 262
merge 8: (262, 99) -> 263
merge 9: (263, 99) -> 264
merge 10: (264, 99) -> 265
merge 11: (265, 99) -> 266
merge 12: (266, 99) -> 267
merge 13: (267, 99) -> 268
merge 14: (268, 99) -> 269
merge 15: (269, 99) -> 270
merge 16: (270, 99) -> 271
merge 17: (271, 99) -> 272
merge 18: (272, 99) -> 273
merge 19: (273, 99) -> 274
merge 20: (274, 99) -> 275


In [16]:
print("final tokens:", tokens)
print("total merges saved:", len(merges))

final tokens: [275, 99]
total merges saved: 20


In [17]:
print(bytes([97]))
print(bytes([104, 101, 108, 108, 111]))

b'a'
b'hello'


In [18]:
vocab = {i: bytes([i]) for i in range(256)}

for (a, b), new_token in merges.items():
    vocab[new_token] = vocab[a] + vocab[b]

print(vocab[256])
print(vocab[257])
print(vocab[258])

b'aa'
b'aaa'
b'aaab'


In [19]:
def decode(ids):
    tokens = b"".join(vocab[i] for i in ids)  # ids.map(i=>vocab[i]).join(b"")
    return tokens.decode("utf-8")              # buffer.toString("utf-8")

In [20]:
print(decode([258, 99]))

aaabc


In [22]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        best_pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if best_pair not in merges:
            break
        new_token = merges[best_pair]
        tokens = merge(tokens, best_pair, new_token)
    return tokens

print(encode("aaab"))    # expect [258]
print(encode("aaabc"))   # expect [258, 99]

[258, 98]
[258, 99]


In [25]:
text = "aaabdaaabac"
tokens = list(text.encode("utf-8"))
print(len(tokens))
print(encode("aaab"))
print(encode("aaabdaaabac"))  # the original training text

11
[258, 98]
[275, 99]


In [26]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-06 02:25:09--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-06 02:25:09 (21.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [27]:
sample = "Hello, how are you?"
print(decode(encode(sample)) == sample)  # should print True

True
